# Transaction exceptions in the original workbook

Use a CPU and run from top to bottom. Inspect three overlapping sets of **original rows**: invoices beginning with `C`, quantities ≤ 0, and unit prices ≤ 0. Exact duplicates and missing customers remain in scope. The earlier five-rule audit simulated sequential exclusions. Here, every comparison starts from the full worksheet. The only output file is an Excel workbook with filterable tables and source row numbers.

# 原始工作簿中的交易异常

使用 CPU，从上到下运行。核查三组可能重叠的**原始行**：发票号以 `C` 开头、数量 ≤ 0、单价 ≤ 0。完全重复行和缺少客户编号的行均参与统计。此前的五规则审计模拟了逐步排除记录；本次各项比较均从完整原表出发。唯一的输出文件是一份 Excel 工作簿，内含可筛选的表格和原始行号。

## Input and output

In Colab, mount Drive and use the original `data/raw/Online Retail.xlsx`. If prompted, copy its path from the Files panel. A workbook under `data/raw/` identifies the project folder automatically. For a different layout, select the project's output folder. Locally, run from the repository root or from `docs/technical/`. Check both printed paths.

## 输入与输出

在 Colab 挂载 Drive，使用原始的 `data/raw/Online Retail.xlsx`。出现路径提示时，从左侧「文件」面板复制工作簿路径。如果它位于 `data/raw/` 下，代码会自动定位项目文件夹。其他目录结构需指定用于存放结果的项目文件夹。本地可从仓库根目录或 `docs/technical/` 运行。请核对打印的两条路径。

In [1]:
from pathlib import Path
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is None:
    project = Path.cwd()
    if project.parts[-2:] == ("docs", "technical"):
        project = project.parents[1]
    workbook = project / "data/raw/Online Retail.xlsx"
else:
    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/CS5481_Project2_OnlineRetail")
    workbook = project / "data/raw/Online Retail.xlsx"
    if not workbook.is_file():
        workbook = Path(input("Full workbook path / 工作簿完整路径：").strip())
        if workbook.parent.name == "raw" and workbook.parent.parent.name == "data":
            project = workbook.parents[2]
        else:
            project = Path(input("Drive project folder / Drive 项目文件夹完整路径：").strip())
            if not project.is_absolute() or not project.is_dir():
                raise ValueError("Select an existing project folder / 请选择已存在的项目文件夹")

if not workbook.is_file():
    raise FileNotFoundError(f"Workbook not found / 找不到工作簿：{workbook}")
output = project / "results/transaction_exceptions.xlsx"
print("Input / 输入：", workbook)
print("Output / 输出：", output)

Input / 输入： /workspace/online-retail-top10-recommendation/data/raw/Online Retail.xlsx
Output / 输出： /workspace/online-retail-top10-recommendation/results/transaction_exceptions.xlsx


## Load the Python source

Local runs import the analysis from the checkout. Colab downloads a ZIP for one specific Git commit and checks each required Python file against its SHA-256 digest. If access requires a browser login, open the printed GitHub link, download the ZIP, and select it in the upload box. The same verification applies.

## 加载 Python 源码

本地运行时从当前仓库导入分析代码。Colab 下载指定 Git 提交的 ZIP，并用 SHA-256 核验所需的每个 Python 文件。如果需要浏览器登录，请打开打印的 GitHub 链接，下载 ZIP 后在上传框选中该文件。手动上传的源码也会进行同样的校验。

In [2]:
import sys

if drive is None:
    source_root = project
else:
    from hashlib import sha256
    from io import BytesIO
    from tempfile import TemporaryDirectory
    from urllib.request import urlopen
    from zipfile import ZipFile

    repository = "NobunagaK/online-retail-top10-recommendation"
    source_commit = "f261bbf0d64edea454e30704fa9169bee6aca429"
    expected_hashes = {
        "scripts/analyze_exceptions.py": "041d80bbb274c605503fb565d3b95ccc4353fe37119d792dd6664d3fb3ff2924",
        "src/data/clean.py": "2dff09028616216f618687e6e2d19c6936c94e7ecbfda87990a4602ed9889412",
        "src/data/exception_impact.py": "a9a4b90f52678d67e3668957093bd66055bbb4fc2a0356b60caae7cd44903704",
        "src/data/exception_missing.py": "ec64dc7f4f61f9da0e88d66cc2549248d3a77ba030ab41ecde9101a977449569",
        "src/data/exception_notes.py": "a81685d6f22efe9cca4ee04728e8909a9d9f39819324dcd8aba875473b74e3bb",
        "src/data/purchase_evidence.py": "0280f0c51753fee7b5c1bcc80b51358d0f519ffb0dadedf46c6ed8cf138998fa",
    }
    archive_url = f"https://github.com/{repository}/archive/{source_commit}.zip"
    try:
        with urlopen(archive_url, timeout=40) as response:
            archive = response.read()
    except OSError:
        from google.colab import files

        print("Download this ZIP in your browser / 在浏览器中下载此 ZIP：", archive_url)
        upload = files.upload()
        if len(upload) != 1:
            raise ValueError("Select the GitHub ZIP / 请选择该 GitHub ZIP 文件")
        archive = next(iter(upload.values()))

    source_directory = TemporaryDirectory()
    source_root = Path(source_directory.name)
    prefix = f"{repository.split('/')[-1]}-{source_commit}/"
    with ZipFile(BytesIO(archive)) as zip_file:
        for relative, digest in expected_hashes.items():
            contents = zip_file.read(prefix + relative)
            if sha256(contents).hexdigest() != digest:
                raise ValueError(f"Source checksum mismatch / 源码校验未通过：{relative}")
            target = source_root / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(contents)
    print("Source commit / 源码提交：", source_commit)

sys.path.insert(0, str(source_root))
from scripts.analyze_exceptions import read_workbook, build_tables, write_workbook

## Count the sign combinations

Read the worksheet once, verify its checksum and build all tables. Each row in `sign_combinations` groups lines by invoice prefix, sign of quantity, sign of unit price, and presence of a customer ID. The groups partition the worksheet. `rows` counts source rows, `invoices` counts distinct invoice numbers within that group, and `signed_gbp` sums quantity × unit price. One invoice may appear in several groups.

## 统计符号组合

读取一次工作表，校验文件哈希并构建全部结果表。`sign_combinations` 按发票前缀、数量符号、单价符号和是否有客户编号分组，每条原始行恰好落入一个组合。`rows` 是原始行数，`invoices` 是组内不同发票号数，`signed_gbp` 为数量乘以单价后的带符号金额之和。同一张发票可以出现在多个组。

In [3]:
raw = read_workbook(workbook)
tables = build_tables(raw)
combinations = tables["sign_combinations"]
assert combinations["rows"].sum() == len(raw)
display(combinations)

,invoice_c,quantity_sign,price_sign,customer_present,rows,invoices,stock_codes,signed_gbp
0,False,negative,zero,False,1336,1336,1082,0.000
1,False,positive,negative,False,2,2,1,-22124.120
2,False,positive,positive,False,132220,1428,3408,1755276.640
3,False,positive,positive,True,397884,18532,3665,8911407.904
4,False,positive,zero,False,1139,785,654,0.000
5,False,positive,zero,True,40,34,35,0.000
6,True,negative,positive,False,383,182,161,-285470.400
7,True,negative,positive,True,8905,3654,1920,-611342.090


## Missing fields and competing filters

Read `missingness` before deciding how a field can be used. A customer ID identifies a person. Averaging customer numbers or inventing an ID would combine unrelated histories. Keep unidentified rows for this raw-data investigation and count them separately. The project excludes them when it constructs personalized purchase histories. A row with a valid StockCode can still identify a product when Description is missing. Use the code as its display label when a suitable description is unavailable. The invoice table lets you inspect which invoice numbers are affected by absent customer IDs.

The `filter_attribution` table revisits the earlier hypothetical five-rule filter in its specified order. For the 10,624 rows with nonpositive quantity, 37 are first accounted for by duplicates, 1,715 by missing customer ID, and 8,872 by the C-invoice flag. Thus quantity contributes zero additional removals at its step. The final price rule accounts for 40 additional rows, all with zero unit price. These figures describe overlap and rule order. All original rows still feed the exception tables below.

All 135,080 missing customer IDs occur on 3,710 invoices. Across the complete sheet, none of those invoices contains a known customer on another line. All 1,454 missing descriptions also lack a customer ID. Quantity and UnitPrice have no missing values. A zero price is an observed value. Lecture 3, slide 10, lists imputation as one possible response to missing data. Here a mean cannot identify a buyer or establish a paid price. [Lecture 3](../../resource/Lecture_3.pdf)

## 缺失字段与相互重叠的规则

先查看 `missingness`，再判断每个字段如何用于分析。客户编号用于识别客户，对编号取均值或虚构编号会把无关历史混在一起。本次原始数据核查保留无编号行并单独计数，项目在构建个性化购买历史时才剔除这些行。如果 StockCode 有效，即使缺少 Description 也能识别商品；没有合适的商品名称时，可直接用商品编码展示。发票表可用来定位哪些发票号涉及客户编号缺失。

`filter_attribution` 按此前假设的五规则顺序追溯净排除量。在 10,624 条非正数量行中，37 条首先归入完全重复，1,715 条首先归入客户编号缺失，8,872 条首先归入 C 发票。因此数量规则本身新增排除 0 行。最后的价格规则新增排除 40 行，均为零单价。以上是规则重叠及顺序产生的归因，后续异常表仍使用所有原始行。

缺客户编号的 135,080 行涉及 3,710 张发票。回查整张原表，这些发票的其他明细也都没有已知客户编号。缺商品描述的 1,454 行全部同时缺客户编号。数量和单价都没有空值，零价是原表实际记录的值。Lecture 3 第 10 页列出多种缺失值处理方式；在此，对编号求均值无法识别买家，用均价替代零也无法证明付费。[Lecture 3](../../resource/Lecture_3.pdf)

In [4]:
display(tables["missingness"])
display(tables["filter_attribution"])
missing_id_invoices = tables["customer_id_invoices"]
print(f"Affected invoices / 涉及的发票：{len(missing_id_invoices):,}")
display(missing_id_invoices.head(10))

,field,raw_rows,missing_rows,missing_pct,invoices_with_missing,missing_with_missing_customer_id
0,InvoiceNo,541909,0,0.000000,0,0
1,StockCode,541909,0,0.000000,0,0
2,Description,541909,1454,0.268311,1454,1454
3,Quantity,541909,0,0.000000,0,0
4,InvoiceDate,541909,0,0.000000,0,0
5,UnitPrice,541909,0,0.000000,0,0
6,CustomerID,541909,135080,24.926694,3710,135080
7,Country,541909,0,0.000000,0,0


,scope,raw_rows,exact_duplicate,missing_customer_id,cancelled_invoice,nonpositive_quantity,nonpositive_unit_price,retained
0,all_rows,541909,5268,135037,8872,0,40,392692
1,C_invoices,9288,37,379,8872,0,0,0
2,Q_nonpositive,10624,37,1715,8872,0,0,0
3,P_nonpositive,2517,5,2472,0,0,40,0
4,zero_quantity,0,0,0,0,0,0,0
5,negative_quantity,10624,37,1715,8872,0,0,0
6,zero_price,2515,5,2470,0,0,40,0
7,negative_price,2,0,2,0,0,0,0
8,missing_customer_id,135080,43,135037,0,0,0,0
9,missing_description,1454,0,1454,0,0,0,0


Affected invoices / 涉及的发票：3,710


,InvoiceNo,first_excel_row,first_missing_excel_row,total_rows,missing_customer_rows,known_customer_ids
0,536414,624,624,1,1,0
1,536544,1445,1445,527,527,0
2,536545,1972,1972,1,1,0
3,536546,1973,1973,1,1,0
4,536547,1974,1974,1,1,0
5,536549,1989,1989,1,1,0
6,536550,1990,1990,1,1,0
7,536552,2026,2026,1,1,0
8,536553,2027,2027,1,1,0
9,536554,2028,2028,1,1,0


## Inspect the three sets side by side

The code prints the full size of each set and displays the first six rows in worksheet order. They are examples, not a statistical sample. `ExcelRow` points to the original worksheet including the header. A C-prefix is the dataset's cancellation indicator. A negative quantity records a decrease and a zero price gives a zero recorded amount regardless of quantity. Read the line description and invoice context before attaching a reason. Look at zero and negative values separately. In this workbook no quantity equals zero, 2,515 rows have zero unit price and two have negative unit price. The full sets, with additional evidence columns, are written to Excel.

## 并列检查三组记录

代码打印各组完整行数，并按工作表顺序展示前六行，便于查看实例。这里采用顺序展示，不能据此估计比例。`ExcelRow` 对应原始工作表的行号，首行为表头。C 前缀是数据集的取消标记。负数量记录数量减少，单价为零时，无论数量多少，行金额均为零。解释原因时还要查看描述与同票记录。数量、单价的零值和负值分别统计。该工作簿中数量为零有 0 行，单价为零有 2,515 行，单价为负有 2 行。完整记录及更多证据列均写入 Excel。

In [5]:
line_fields = [
    "ExcelRow", "InvoiceNo", "InvoiceDate", "StockCode", "Description",
    "Quantity", "UnitPrice", "CustomerID", "amount_gbp",
]
for name in ["C_invoices", "Q_nonpositive", "P_nonpositive"]:
    group = tables[name]
    print(f"{name} · {len(group):,} rows / 行")
    display(group[line_fields].head(6))

display(tables["P_nonpositive"].loc[
    tables["P_nonpositive"]["price_sign"].eq("negative"), line_fields
])

C_invoices · 9,288 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
141,143,C536379,2010-12-01 09:41:00,D,Discount,-1,27.50,14527,-27.50
154,156,C536383,2010-12-01 09:49:00,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,15311,-4.65
235,237,C536391,2010-12-01 10:24:00,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,17548,-19.80
236,238,C536391,2010-12-01 10:24:00,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,17548,-6.96
237,239,C536391,2010-12-01 10:24:00,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,17548,-6.96
238,240,C536391,2010-12-01 10:24:00,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,17548,-6.96


Q_nonpositive · 10,624 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
141,143,C536379,2010-12-01 09:41:00,D,Discount,-1,27.50,14527,-27.50
154,156,C536383,2010-12-01 09:49:00,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,15311,-4.65
235,237,C536391,2010-12-01 10:24:00,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,17548,-19.80
236,238,C536391,2010-12-01 10:24:00,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,17548,-6.96
237,239,C536391,2010-12-01 10:24:00,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,17548,-6.96
238,240,C536391,2010-12-01 10:24:00,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,17548,-6.96


P_nonpositive · 2,517 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
622,624,536414,2010-12-01 11:52:00,22139,NaN,56,0.0,<NA>,0.0
1970,1972,536545,2010-12-01 14:32:00,21134,NaN,1,0.0,<NA>,0.0
1971,1973,536546,2010-12-01 14:33:00,22145,NaN,1,0.0,<NA>,0.0
1972,1974,536547,2010-12-01 14:33:00,37509,NaN,1,0.0,<NA>,0.0
1987,1989,536549,2010-12-01 14:34:00,85226A,NaN,1,0.0,<NA>,0.0
1988,1990,536550,2010-12-01 14:34:00,85044,NaN,1,0.0,<NA>,0.0


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
299983,299985,A563186,2011-08-12 14:51:00,B,Adjust bad debt,1,-11062.06,<NA>,-11062.06
299984,299986,A563187,2011-08-12 14:52:00,B,Adjust bad debt,1,-11062.06,<NA>,-11062.06


## Negative quantities on non-C invoices and earlier positive quantities

Start with `nonC_negative`. The 1,336 lines each occupy a single-line invoice, have zero unit price and lack a customer ID. Of these, 862 also lack a description. Use `history_summary` to compare earlier positive-quantity lines for the exact invoice, invoice + SKU, SKU alone, and customer + SKU. Earlier means a strictly earlier timestamp. The positive-quantity lookups include every earlier row with Quantity > 0 regardless of price or missing ID. The stronger customer/SKU comparison also requires a known customer and a non-C positive-price purchase line. A blank customer match is unavailable evidence. A previous occurrence of the SKU shows the item was recorded before. Matching the customer, price, quantity magnitude, and time order provides stronger evidence of a possible reversal. Even that match alone does not verify a physical return or refund.

For the 1,336 lines, the exact-invoice and exact-invoice-plus-SKU searches each find zero earlier positive lines. The weaker SKU-only search succeeds on 1,179 lines. For the 9,288 C lines, removing the C and looking for that invoice number plus SKU finds zero earlier positive matches. A different search finds 2,816 C lines with at least one earlier non-C, positive-price line for the same customer, SKU, unit price and opposite quantity. These are candidate lines, potentially several per credit. The workbook contains no allocation of each credit to an original sale.

## 非 C 发票的负数量与此前的正数量记录

从 `nonC_negative` 入手。这 1,336 行每行各占一张仅有一行的发票，单价为零、客户编号缺失，其中 862 行还缺少商品描述。通过 `history_summary` 分别比较同发票、同发票与商品、同商品，以及同客户与商品的早先正数量记录。“早先”要求时间戳严格早于当前行。按正数量回查时，单价为零或缺客户编号的早期正数量行也参与。核对更强的同客户、同商品候选时，另要求客户编号已知，且候选购买行单价为正、发票号不以 C 开头。缺少客户编号时，客户匹配结果为空，表示无法核对。商品编码曾出现，只说明原表中此前记录过这个商品。如果客户、单价、数量绝对值和时间顺序都吻合，则更支持可能存在冲销；仅凭匹配仍无法确认实物退回或款项退回。

对 1,336 行按原发票号、以及按原发票号加商品编码查找，两个口径都没有找到此前的正数量行。放宽到仅按商品编码匹配，则有 1,179 行找到早期记录。对 9,288 条 C 明细去掉 C 前缀后再按发票号加商品回查，也没有早期匹配。换一种方法，按同一客户、同一商品、同一单价、数量恰好相反四个条件查找，2,816 行有更早的非 C、正价候选购买行。一条贷项可能对应多个候选，原表没有给出逐笔核销关系。

In [6]:
non_c = tables["nonC_negative"]
print("Missing descriptions / 缺失描述：", non_c["Description"].isna().sum())
display(non_c[
    line_fields + ["invoice_rows_full_sheet", "earlier_positive_invoice_rows",
                   "earlier_positive_invoice_sku_rows", "earlier_positive_sku_rows",
                   "earlier_positive_customer_sku_rows"]
].head(8))
display(tables["history_summary"])

Missing descriptions / 缺失描述： 862


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp,invoice_rows_full_sheet,earlier_positive_invoice_rows,earlier_positive_invoice_sku_rows,earlier_positive_sku_rows,earlier_positive_customer_sku_rows
2406,2408,536589,2010-12-01 16:50:00,21777,NaN,-10,0.0,<NA>,-0.0,1,0,0,3,<NA>
4347,4349,536764,2010-12-02 14:42:00,84952C,NaN,-38,0.0,<NA>,-0.0,1,0,0,0,<NA>
7188,7190,536996,2010-12-03 15:30:00,22712,NaN,-20,0.0,<NA>,-0.0,1,0,0,4,<NA>
7189,7191,536997,2010-12-03 15:30:00,22028,NaN,-20,0.0,<NA>,-0.0,1,0,0,4,<NA>
7190,7192,536998,2010-12-03 15:30:00,85067,NaN,-6,0.0,<NA>,-0.0,1,0,0,1,<NA>
7192,7194,537000,2010-12-03 15:32:00,21414,NaN,-22,0.0,<NA>,-0.0,1,0,0,0,<NA>
7193,7195,537001,2010-12-03 15:33:00,21653,NaN,-6,0.0,<NA>,-0.0,1,0,0,0,<NA>
7195,7197,537003,2010-12-03 15:33:00,85126,NaN,-2,0.0,<NA>,-0.0,1,0,0,0,<NA>


,cohort,rows,distinct_invoices,singleton_invoice_rows,prior_same_invoice_rows_matched,prior_same_invoice_sku_rows_matched,prior_sku_rows_matched,customer_sku_comparable_rows,prior_customer_sku_rows_matched,prior_stripped_invoice_sku_rows_matched,prior_customer_sku_price_opposite_units_rows_matched,same_invoice_has_positive_priced_rows
0,any_exception,11805,5993,4229,0,0,11391,8945,7627,0,2816,400
1,C_invoices,9288,3836,2131,0,0,9102,8905,7599,0,2816,0
2,nonC_negative,1336,1336,1336,0,0,1179,0,0,0,0,0
3,zero_qty,0,0,0,0,0,0,0,0,0,0,0
4,positive_qty_zero_price,1179,819,760,0,0,1108,40,28,0,0,400
5,negative_price,2,2,2,0,0,2,0,0,0,0,0


## Read the description, item code, and invoice reference

The note summary counts literal text and special-code signals by group. `blank` means there was no description to interpret; `unknown` means the text did not match a recognized note. Inspect a note and its referenced document as separate pieces of evidence. For instance, invoices `554550` and `554553` cite `C550456` and contain “incorrectly credited”. Invoice `554551` says “reverse previous adjustment”. The signs and text support an adjustment sequence. The workbook does not identify a corresponding shipment. For `BATHROOM METAL SIGN`, the two raw descriptions differ in trailing whitespace. The name lookup trims whitespace only to find the example and retains both stock codes. Similar descriptions cannot establish an SKU alias.

For example, the description for invoice 554143 says “damages”, while 573482 says only “check”. The first is evidence of a damage notation. The second leaves the result of that check unknown. On the zero-price rows, 1,179 have positive quantity. Among those, 40 have a customer ID and 1,139 do not. Distinguishing gifts from stock entries or a missing charge requires further documentation.

## 核查原始描述、商品编码和备注引用的发票

备注汇总按组统计字面文字和特殊商品码信号。`blank` 表示没有描述可供解释；`unknown` 表示文本未命中已设置的备注规则。备注内容和它指向的单据可以分别核查。例如 `554550` 与 `554553` 的描述包含 “incorrectly credited”，并提及 `C550456`；`554551` 则写有 “reverse previous adjustment”。正负号和描述共同支持存在调整的解释，表内没有标明对应的发货记录。`BATHROOM METAL SIGN` 的两条原始描述在末尾空格上存在差别。名称查找只为定位示例去除首尾空格，同时保留两个商品编码。仅凭名称相似无法确定两个编码互为别名。

例如，发票 554143 备注为 “damages”，说明经办记录提到了损坏；573482 只写 “check”，既未说明盘点结果也未说明客户退货。零价行中有 1,179 行数量为正，其中 40 行有客户编号、1,139 行无客户编号。赠品、库存调整或漏记价格等可能原因仍需其他凭证。

In [7]:
notes = tables["note_summary"]
display(notes.loc[notes["scope"].eq("nonC_negative")].sort_values("rows", ascending=False))
display(tables["invoice_references"].head(20))
display(tables["sku_names"])

,scope,description_note_signals,special_code_signal,rows
38,nonC_negative,blank,,862
39,nonC_negative,damage_or_disposal,,171
40,nonC_negative,stock_adjustment_or_count,,141
41,nonC_negative,unknown,,88
42,nonC_negative,set_or_unit_sale_wording,,32
43,nonC_negative,missing_or_lost,,18
44,nonC_negative,item_code_or_label_discrepancy,,14
45,nonC_negative,credit_note_wording,,3
46,nonC_negative,sample_or_promotion,,3
47,nonC_negative,damage_or_disposal|sample_or_promotion,,2


,ExcelRow,InvoiceNo,StockCode,Description,mentioned_invoice_id,referenced_invoice_rows,earlier_positive_quantity_rows,earlier_positive_quantity_sku_rows,earlier_same_sku_rows,earlier_credit_or_nonpositive_rows,earlier_credit_or_nonpositive_sku_rows,example_earlier_same_sku_excel_row
0,203753,554550,47566B,incorrectly credited C550456 see 47,C550456,5,0,0,1,5,1,160144
1,203763,554553,47566B,incorrectly credited C550456 see 47,C550456,5,0,0,1,5,1,160144


,Description,StockCode,workbook_rows,first_excel_row,exact_description_length,distinct_skus_for_exact_description,distinct_skus_for_trimmed_name
0,BATHROOM METAL SIGN,82580,641,343,19,1,2
1,BATHROOM METAL SIGN,21171,73,13319,20,1,2


## Compare the named cases against their source rows

- **A large reversal.** Compare `541431` and `C541433`. The same customer, SKU and unit price appear 16 minutes apart with quantities +74,215 and −74,215. The amounts offset. The two invoice IDs differ, even after removing the C.
- **A discount and an unpriced item.** On `C536379`, SKU `D` has the description “Discount”. A credit can refer to a discount. On `539722`, SKU `22423` has positive quantity and zero price alongside priced items. The recorded zero leaves the reason open, including whether the item was free.
- **Ledger and stock adjustments.** The code `BANK CHARGES` identifies a bank-charge entry. Compare the positive and negative prices of the `B` entries `A563185`–`A563187` (“Adjust bad debt”). These fee and debt records require different interpretations from a merchandise return. Then follow SKU `47566B` through `C550456`, `554550`, `554551`, and `554553`. Invoice `560039` explicitly mentions another code in the box; use the recorded SKU when tracing this row.

The tables show what the workbook records. Customer intent, the cause of a price of zero, physical stock movement, and settlement may require other records.

## 将指定案例与原始行对照

- **大额正负对冲。** 比较 `541431` 与 `C541433`：客户、商品编码、单价相同，相隔 16 分钟，数量分别为 +74,215 和 −74,215，两个行金额相抵。即使去掉 C，发票号也不同。
- **折扣与零价商品。** `C536379` 的商品码为 `D`，描述为 “Discount”（折扣），说明贷项可以涉及折扣。`539722` 同票中还有正常计价行，而 `22423` 数量为正、单价为零；原表留下了零价记录，是否赠品等具体原因仍待核查。
- **账务与库存调整。** 商品码 `BANK CHARGES` 标识银行费用。对照 `A563185` 至 `A563187` 的 `B` 行，查看 “Adjust bad debt”（坏账调整）所对应的正负单价。核查这类费用和坏账时应分别考虑其账务含义，不能据负号把它们当成商品退货。再沿 `47566B` 追踪 `C550456`、`554550`、`554551` 和 `554553`。`560039` 的描述明确提到箱中另一个商品编码；追踪本行时须保留原始商品编码。

表格给出工作簿记载的事实。客户意图、零价的具体原因、实际库存流向和款项结算可能需要其他单据印证。

In [8]:
display(tables["case_examples"][["case", *line_fields]])
case_invoice = raw.loc[raw["InvoiceNo"].eq("539722")]
display(case_invoice.groupby("UnitPrice", as_index=False).agg(
    rows=("ExcelRow", "size"), units=("Quantity", "sum")
))
bank_fees = raw.loc[raw["StockCode"].eq("BANK CHARGES")]
display(bank_fees[["ExcelRow", "InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]].head(4))

,case,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
0,same-customer opposite entries,61621,541431,2011-01-18 10:01:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346,77183.60
1,same-customer opposite entries,61626,C541433,2011-01-18 10:17:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346,-77183.60
2,discount credit,143,C536379,2010-12-01 09:41:00,D,Discount,-1,27.50,14527,-27.50
3,zero recorded unit price,40091,539722,2010-12-21 13:45:00,22423,REGENCY CAKESTAND 3 TIER,10,0.00,14911,0.00
4,bad-debt entries,299984,A563185,2011-08-12 14:50:00,B,Adjust bad debt,1,11062.06,<NA>,11062.06
5,bad-debt entries,299985,A563186,2011-08-12 14:51:00,B,Adjust bad debt,1,-11062.06,<NA>,-11062.06
6,bad-debt entries,299986,A563187,2011-08-12 14:52:00,B,Adjust bad debt,1,-11062.06,<NA>,-11062.06
7,explicit credit note and later adjustments,160144,C550456,2011-04-18 13:08:00,47566B,TEA TIME PARTY BUNTING,-1300,2.55,15749,-3315.00
8,explicit credit note and later adjustments,203753,554550,2011-05-25 09:57:00,47566B,incorrectly credited C550456 see 47,1300,0.00,<NA>,0.00
9,explicit credit note and later adjustments,203754,554551,2011-05-25 09:58:00,47566B,reverse previous adjustment,-1300,0.00,<NA>,-0.00


,UnitPrice,rows,units
0,0.00,1,10
1,0.55,2,120
2,0.85,1,24
3,1.65,2,22
4,1.95,1,10
5,2.10,4,66
6,2.95,5,48
7,3.75,1,8
8,4.25,1,6
9,4.65,1,8


,ExcelRow,InvoiceNo,StockCode,Description,Quantity,UnitPrice
4406,4408,536779,BANK CHARGES,Bank Charges,1,15.00
14435,14437,C537572,BANK CHARGES,Bank Charges,-1,95.38
28992,28994,C538680,BANK CHARGES,Bank Charges,-1,966.92
62508,62510,541505,BANK CHARGES,Bank Charges,1,15.00


## Amounts and customer activity under two definitions

Compare all original lines with lines on non-C invoices having positive quantity and positive unit price. The latter is a descriptive baseline for this calculation. It may still contain fees or other non-merchandise codes. The first table separates identified and unidentified customers and reports positive, negative, and signed line amounts. The second groups identified customers by whether they have regular entries, exceptions, or both. Recency is measured from the day after the latest workbook date. Compare days since **any activity** with days since a **regular purchase**. The signed amount is an arithmetic result, not evidence that payment was collected.

Chen, Sain and Guo study calendar 2011 and group UK transactions by postcode. They compute a line amount as quantity times price and isolate unusual customer profiles for separate examination before clustering (pp. 199–203). Our eight-column workbook has no postcode. We adapt their recency, distinct-invoice frequency and monetary measures to observed CustomerID over the workbook window and compare two definitions of activity. Among 1,563 customers with both kinds of lines, the median recency is 30 days for regular lines and 26 days when exceptional entries count as activity. Another 34 customers have exceptional entries only. These figures describe observed activity and cannot be used as cutoff-safe features for training. The paper supplies no transaction-level return or zero-price label.

## 两种统计口径下的金额和客户活动

对比全部原始行与“非 C 发票、数量为正且单价为正”的行。后者用作本次比较基准，其中仍可能含有费用或其他非商品码。第一张表区分客户编号是否缺失，分别统计正、负行金额及带符号金额合计。第二张表按是否有常规记录、异常记录或两者都有，对可识别客户分组。距最近活动天数均以工作簿最后一个日期的次日为基准，分别计算距**任意活动**和距**常规购买**的天数。带符号金额来自数量乘以单价，实际收款情况需另行核实。

Chen、Sain 与 Guo 研究 2011 自然年的英国交易，按邮编汇总，以数量乘单价计算行金额，并将异常客户画像单独审视后再聚类（第 199–203 页）。我们的八列表没有邮编，因此改用已有 CustomerID，按同一数据窗口比较距最近记录的天数、不同发票数和金额。1,563 位同时有两类记录的客户中，只看常规行时距最近记录天数的中位数为 30 天，计入特殊行后为 26 天。另有 34 位客户只出现于特殊行。这是全期描述；训练特征仍需按各自时间截点重算。论文没有为单笔记录提供退货或零价成因标签。

In [9]:
display(tables["amount_by_record_type"])
display(tables["customer_impact_summary"])

,record_type,customer_id_status,rows,distinct_invoices,positive_line_gbp,negative_line_gbp,signed_gbp
0,regular,identified,397884,18532,8911407.904,0.00,8911407.904
1,regular,missing,132220,1428,1755276.640,0.00,1755276.640
2,exception,identified,8945,3688,0.000,-611342.09,-611342.090
3,exception,missing,2860,2305,0.000,-307594.52,-307594.520


,cohort,customers,regular_gross_gbp,all_signed_gbp,amount_difference_gbp,median_regular_recency_days,median_all_recency_days
0,regular_only,2775,2285576.522,2285576.522,0.00,65.0,65.0
1,mixed,1563,6625831.382,6024900.532,-600930.85,30.0,26.0
2,exception_only,34,0.000,-10411.240,-10411.24,<NA>,346.5


## Export and sources

Write all tables to `results/transaction_exceptions.xlsx`. In Excel, filter by `ExcelRow`, `InvoiceNo`, or `StockCode` to trace a row. The original eight fields are present in the full case sheets. Chen, Sain and Guo (2012), pp. 199–203, discuss transaction preparation and RFM customer segmentation. Their analysis supplies context for the customer-activity comparison. The counts and case evidence here come directly from the identified workbook. [Chen, Sain & Guo (2012)](https://doi.org/10.1057/dbm.2012.17) · [Chen (2015), UCI Online Retail, CC BY 4.0](https://doi.org/10.24432/C5BW33)

## 导出与来源

将全部表格写入 `results/transaction_exceptions.xlsx`。在 Excel 中可用 `ExcelRow`、`InvoiceNo` 或 `StockCode` 筛选和追溯明细，完整案例工作表保留八个原始字段。Chen、Sain 和 Guo（2012，第 199–203 页）讨论了交易数据准备与基于 RFM 的客户分群，为此次客户活动口径比较提供背景。本次统计量和案例证据均取自已校验的原始工作簿。[Chen、Sain 和 Guo（2012）](https://doi.org/10.1057/dbm.2012.17) · [Chen（2015），UCI Online Retail，CC BY 4.0](https://doi.org/10.24432/C5BW33)

In [10]:
write_workbook(tables, output)
print(f"Saved {len(tables)} sheets / 已保存 {len(tables)} 张工作表：{output}")

Saved 17 sheets / 已保存 17 张工作表：/workspace/online-retail-top10-recommendation/results/transaction_exceptions.xlsx
